# FASE 2: PIPELINE DE PROCESAMIENTO DE DATOS CON PYSPARK (LOCAL)

## 📌 Contexto del Proyecto
En la Fase 1 (EDA), analizamos visualmente el comportamiento de los clientes de la empresa Telco. Descubrimos que la pérdida de clientes (*Churn*) está fuertemente ligada a factores como el tipo de contrato mensual y cargos acumulados altos. También detectamos problemas de calidad de datos, específicamente espacios en blanco en la columna `TotalCharges` que bloqueaban su uso matemático.

## 🎯 Objetivo de este Notebook
El objetivo de este módulo es construir un **Pipeline (Tubería) de Procesamiento de Datos** automatizado utilizando **PySpark**. Este pipeline se encargará de transformar los datos crudos del negocio en un formato estructurado y optimizado para los algoritmos de Machine Learning, corriendo de forma distribuida en nuestro entorno local.

## ⚙️ Estructura de la Tubería (Pipeline)
El procesamiento se divide en 4 estaciones secuenciales:
1. **Ingesta :** Conexión y lectura del dataset mapeando los tipos de datos iniciales.
2. **Limpieza Automática:** Conversión forzada de `TotalCharges` a tipo numérico y rellenado inteligente de valores nulos utilizando la mediana del negocio.
3. **Indexación Categórica (Feature Engineering):** Traducción de variables de texto (como género, contratos y servicios) a índices numéricos manejables mediante `StringIndexer`.
4. **Ensamblado Vectorial:** Agrupación de todas las variables predictoras en un único vector de características (`features`) exigido por PySpark ML.

In [1]:
# ==============================================================================
# 1. CONFIGURACIÓN DE LA SESIÓN DE PYSPARK
# ==============================================================================
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, round

# Crear la sesión de Spark optimizada para Big Data local
spark = SparkSession.builder \
    .appName("TelcoChurn_DataPipeline") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

# Verificar que Spark esté corriendo
print(f"Versión de PySpark activa: {spark.version}")

Versión de PySpark activa: 3.5.0


In [2]:
# ==============================================================================
# 2. INGESTA DE DATOS (LECTURA DEL DATASET)
# ==============================================================================
# Definir la ruta relativa al archivo CSV
csv_path = "../data/sample/WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Leer el CSV con PySpark infiriendo el esquema y detectando la cabecera
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

# Mostrar cuántos registros y columnas se cargaron
print(f"Dataset cargado con éxito.")
print(f"Total de filas: {df.count()}")
print(f"Total de columnas: {len(df.columns)}")

# Imprimir el esquema para revisar los tipos de datos iniciales
df.printSchema()

Dataset cargado con éxito.
Total de filas: 7043
Total de columnas: 21
root
 |-- customerID: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- SeniorCitizen: integer (nullable = true)
 |-- Partner: string (nullable = true)
 |-- Dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- PhoneService: string (nullable = true)
 |-- MultipleLines: string (nullable = true)
 |-- InternetService: string (nullable = true)
 |-- OnlineSecurity: string (nullable = true)
 |-- OnlineBackup: string (nullable = true)
 |-- DeviceProtection: string (nullable = true)
 |-- TechSupport: string (nullable = true)
 |-- StreamingTV: string (nullable = true)
 |-- StreamingMovies: string (nullable = true)
 |-- Contract: string (nullable = true)
 |-- PaperlessBilling: string (nullable = true)
 |-- PaymentMethod: string (nullable = true)
 |-- MonthlyCharges: double (nullable = true)
 |-- TotalCharges: string (nullable = true)
 |-- Churn: string (nullable = true)



In [3]:
# ==============================================================================
# 3. LIMPIEZA AUTOMÁTICA DE DATOS (DATA CLEANING)
# ==============================================================================
from pyspark.sql.functions import median

# Paso A: Limpiar espacios en blanco y forzar la columna a tipo numérico (Double)
# Si encuentra un espacio vacío, PySpark lo convertirá automáticamente en "Null"
df_cleaned = df.withColumn(
    "TotalCharges", 
    col("TotalCharges").cast("double")
)

# Paso B: Calcular la mediana de TotalCharges para rellenar los valores nulos de forma inteligente
mediana_total_charges = df_cleaned.approxQuantile("TotalCharges", [0.5], 0.001)[0]
print(f"-> La mediana calculada para TotalCharges es: {mediana_total_charges}")

# Paso C: Rellenar los valores nulos con esa mediana
df_cleaned = df_cleaned.na.fill({"TotalCharges": mediana_total_charges})

# Paso D: Verificar que ya no queden nulos en las columnas clave
print("\nConteo de valores nulos actualizados por columna:")
df_cleaned.select([count(when(col(c).isNull(), c)).alias(c) for c in ["MonthlyCharges", "TotalCharges"]]).show()

-> La mediana calculada para TotalCharges es: 1394.55

Conteo de valores nulos actualizados por columna:
+--------------+------------+
|MonthlyCharges|TotalCharges|
+--------------+------------+
|             0|           0|
+--------------+------------+



In [4]:
# ==============================================================================
# 4. TRANSFORMACIÓN DE CATEGORÍAS A NÚMEROS (FEATURE ENGINEERING)
# ==============================================================================
from pyspark.ml.feature import StringIndexer

# 1. Identificar de forma automática todas las columnas que son de tipo texto (String)
# Excluimos 'customerID' porque es un identificador único, no una característica de negocio
columnas_categoricas = [item[0] for item in df_cleaned.dtypes if item[1] == 'string' and item[0] != 'customerID']

# 2. Crear una lista de indexadores (uno para cada columna de texto)
# El 'outputCol' llevará el mismo nombre de la columna original pero con el sufijo '_index'
indexers = [
    StringIndexer(inputCol=col_name, outputCol=f"{col_name}_index", handleInvalid="keep")
    for col_name in columnas_categoricas
]

# 3. Aplicar las transformaciones en cadena sobre nuestro DataFrame limpio
df_indexed = df_cleaned
for indexer in indexers:
    df_indexed = indexer.fit(df_indexed).transform(df_indexed)

# 4. Mostrar una muestra de cómo quedaron las columnas originales vs las nuevas numéricas
print("Muestra de la transformación categórica (Original vs Indexada):")
df_indexed.select("gender", "gender_index", "Contract", "Contract_index", "Churn", "Churn_index").show(5)

Muestra de la transformación categórica (Original vs Indexada):
+------+------------+--------------+--------------+-----+-----------+
|gender|gender_index|      Contract|Contract_index|Churn|Churn_index|
+------+------------+--------------+--------------+-----+-----------+
|Female|         1.0|Month-to-month|           0.0|   No|        0.0|
|  Male|         0.0|      One year|           2.0|   No|        0.0|
|  Male|         0.0|Month-to-month|           0.0|  Yes|        1.0|
|  Male|         0.0|      One year|           2.0|   No|        0.0|
|Female|         1.0|Month-to-month|           0.0|  Yes|        1.0|
+------+------------+--------------+--------------+-----+-----------+
only showing top 5 rows



In [5]:
# ==============================================================================
# 5. ENSAMBLADO DE CARACTERÍSTICAS 
# ==============================================================================
from pyspark.ml.feature import VectorAssembler

# 1. Definir qué columnas numéricas queremos que el modelo analice
# Incluimos las numéricas originales limpias y las nuevas columnas indexadas
columnas_features = [
    "SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges",
    "gender_index", "Partner_index", "Dependents_index", "PhoneService_index",
    "MultipleLines_index", "InternetService_index", "OnlineSecurity_index",
    "OnlineBackup_index", "DeviceProtection_index", "TechSupport_index",
    "StreamingTV_index", "StreamingMovies_index", "Contract_index",
    "PaperlessBilling_index", "PaymentMethod_index"
]

# 2. Configurar el ensamblador para juntar todo en la columna "features"
assembler = VectorAssembler(
    inputCols=columnas_features, 
    outputCol="features",
    handleInvalid="skip"
)

# 3. Transformar el dataset final
df_final = assembler.transform(df_indexed)

# 4. Seleccionar solo lo que necesita la Inteligencia Artificial: Features y la Meta (Label)
df_vectorizado = df_final.select("features", col("Churn_index").alias("label"))

print("¡Pipeline completado con éxito en local!")
print("Estructura final del dataset listo para Machine Learning:")
df_vectorizado.show(5, truncate=False)

¡Pipeline completado con éxito en local!
Estructura final del dataset listo para Machine Learning:
+-----------------------------------------------------------------------------------+-----+
|features                                                                           |label|
+-----------------------------------------------------------------------------------+-----+
|(19,[1,2,3,4,5,7,8,9,11],[1.0,29.85,29.85,1.0,1.0,1.0,2.0,1.0,1.0])                |0.0  |
|(19,[1,2,3,9,10,12,16,17,18],[34.0,56.95,1889.5,1.0,1.0,1.0,2.0,1.0,1.0])          |0.0  |
|(19,[1,2,3,9,10,11,18],[2.0,53.85,108.15,1.0,1.0,1.0,1.0])                         |1.0  |
|[0.0,45.0,42.3,1840.75,0.0,0.0,0.0,1.0,2.0,1.0,1.0,0.0,1.0,1.0,0.0,0.0,2.0,1.0,2.0]|0.0  |
|(19,[1,2,3,4],[2.0,70.7,151.65,1.0])                                               |1.0  |
+-----------------------------------------------------------------------------------+-----+
only showing top 5 rows

